In [1]:
from pathlib import Path
import pandas as pd

ROOT: Path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_SBDH: Path = ROOT / "data/raw/mimic/MIMIC3_SBDH_2021_Ahsan_rev/MIMIC-SBDH.csv"
IDS: Path = ROOT / "data/interim/sbdh_note_ids.csv"

for p in (RAW_SBDH, IDS):
    print(f"{'OK ' if p.exists() else 'MISSING'} {p.relative_to(ROOT)}")

sbdh: pd.DataFrame = pd.read_csv(RAW_SBDH)
ids: pd.DataFrame = pd.read_csv(IDS)

# shape and integrity checks only, no note text printed
print(f"\nsbdh: {sbdh.shape[0]:,} rows x {sbdh.shape[1]} cols, nulls={int(sbdh.isna().sum().sum())}, dup ROW_ID={int(sbdh['ROW_ID'].duplicated().sum())}")
print(f"ids:  {ids.shape[0]:,} rows, patients={ids['SUBJECT_ID'].nunique():,}")
print(f"regex-fail rows (social history == full note): {int((sbdh['TEXT'] == sbdh['SOCIAL_HISTORY_TEXT']).sum())}")
print(f"full notes truncated at 32,767 chars: {int((sbdh['TEXT'].str.len() == 32767).sum())}")

OK  data\raw\mimic\MIMIC3_SBDH_2021_Ahsan_rev\MIMIC-SBDH.csv
OK  data\interim\sbdh_note_ids.csv

sbdh: 7,025 rows x 11 cols, nulls=0, dup ROW_ID=0
ids:  7,025 rows, patients=6,199
regex-fail rows (social history == full note): 164
full notes truncated at 32,767 chars: 28


In [2]:
LABEL_COLS: list[str] = [
    "sdoh_community_present", "sdoh_community_absent", "sdoh_education",
    "sdoh_economics", "sdoh_environment",
    "behavior_alcohol", "behavior_tobacco", "behavior_drug",
]

df: pd.DataFrame = sbdh.merge(ids, on="ROW_ID", how="left", validate="one_to_one")
df["regex_fail"] = df["TEXT"] == df["SOCIAL_HISTORY_TEXT"]
df["sh_len"] = df["SOCIAL_HISTORY_TEXT"].str.len()

df = df[["ROW_ID", "SUBJECT_ID", "HADM_ID", "SOCIAL_HISTORY_TEXT", "regex_fail", "sh_len"] + LABEL_COLS]

print(f"rows: {len(df):,} | unmatched SUBJECT_ID: {int(df['SUBJECT_ID'].isna().sum())}")
print(f"columns ({df.shape[1]}): {list(df.columns)}")
print(f"regex_fail: {int(df['regex_fail'].sum())}")
print(f"sh_len median: {int(df['sh_len'].median())} | p95: {int(df['sh_len'].quantile(0.95))} | max: {int(df['sh_len'].max())}")
print(f"sh_len median, regex_fail only: {int(df.loc[df['regex_fail'], 'sh_len'].median())}")

rows: 7,025 | unmatched SUBJECT_ID: 0
columns (14): ['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'SOCIAL_HISTORY_TEXT', 'regex_fail', 'sh_len', 'sdoh_community_present', 'sdoh_community_absent', 'sdoh_education', 'sdoh_economics', 'sdoh_environment', 'behavior_alcohol', 'behavior_tobacco', 'behavior_drug']
regex_fail: 164
sh_len median: 251 | p95: 764 | max: 22406
sh_len median, regex_fail only: 6747


In [3]:
EXCLUDE_REGEX_FAIL: bool = True

model_df: pd.DataFrame = df.loc[~df["regex_fail"]].copy() if EXCLUDE_REGEX_FAIL else df.copy()

print(f"modeling rows: {len(model_df):,} (excluded {len(df) - len(model_df)}) | patients: {model_df['SUBJECT_ID'].nunique():,}")
print(f"sh_len max after exclusion: {int(model_df['sh_len'].max())}")
for col in LABEL_COLS:
    print(f"{col:24s} {model_df[col].value_counts().sort_index().to_dict()}")

modeling rows: 6,861 (excluded 164) | patients: 6,064
sh_len max after exclusion: 5128
sdoh_community_present   {0: 2489, 1: 4372}
sdoh_community_absent    {0: 6089, 1: 772}
sdoh_education           {0: 6655, 1: 206}
sdoh_economics           {0: 4177, 1: 968, 2: 1716}
sdoh_environment         {0: 2509, 1: 4262, 2: 90}
behavior_alcohol         {0: 1600, 1: 2028, 2: 508, 3: 2402, 4: 323}
behavior_tobacco         {0: 1233, 1: 994, 2: 2075, 3: 2209, 4: 350}
behavior_drug            {0: 4190, 1: 205, 2: 219, 3: 2105, 4: 142}


In [4]:
from sklearn.model_selection import StratifiedGroupKFold

def rare_key(row: pd.Series) -> str:
    # priority order: rarest first, so each note gets one stratum
    if row["sdoh_environment"] == 2:
        return "homeless"
    if row["behavior_drug"] == 1:
        return "drug_present"
    if row["sdoh_education"] == 1:
        return "education"
    return "other"

model_df["strata"] = model_df.apply(rare_key, axis=1)

sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
model_df["fold"] = -1
for fold_id, (_, idx) in enumerate(sgkf.split(model_df, model_df["strata"], groups=model_df["SUBJECT_ID"])):
    model_df.iloc[idx, model_df.columns.get_loc("fold")] = fold_id

model_df["split"] = pd.cut(model_df["fold"], bins=[-1, 2, 5, 19], labels=["test", "dev", "train"]).astype(str)

# leakage check: every patient in exactly one split
leak: int = int((model_df.groupby("SUBJECT_ID")["split"].nunique() > 1).sum())
print(f"patients in >1 split: {leak}")
print(model_df["split"].value_counts().reindex(["train", "dev", "test"]).to_dict())
print(pd.crosstab(model_df["strata"], model_df["split"])[["train", "dev", "test"]])

patients in >1 split: 0
{'train': 4803, 'dev': 1029, 'test': 1029}
split         train  dev  test
strata                        
drug_present    135   28    28
education       129   29    28
homeless         64   13    13
other          4475  959   960


In [5]:
OUT: Path = ROOT / "data/processed/sbdh_splits.csv"
OUT.parent.mkdir(parents=True, exist_ok=True)

save_cols: list[str] = ["ROW_ID", "SUBJECT_ID", "HADM_ID", "split", "strata", "sh_len", "SOCIAL_HISTORY_TEXT"] + LABEL_COLS
model_df[save_cols].to_csv(OUT, index=False, encoding="utf-8")

check: pd.DataFrame = pd.read_csv(OUT)
same_text: bool = (check["SOCIAL_HISTORY_TEXT"].str.len().values == model_df["SOCIAL_HISTORY_TEXT"].str.len().values).all()
same_labels: bool = (check[LABEL_COLS].values == model_df[LABEL_COLS].values).all()

print(f"saved: {OUT.relative_to(ROOT)} | {OUT.stat().st_size / 1e6:.1f} MB")
print(f"reloaded rows: {len(check):,} | cols: {check.shape[1]} | text intact: {same_text} | labels intact: {same_labels}")
print(check["split"].value_counts().reindex(["train", "dev", "test"]).to_dict())

saved: data\processed\sbdh_splits.csv | 2.4 MB
reloaded rows: 6,861 | cols: 15 | text intact: True | labels intact: True
{'train': 4803, 'dev': 1029, 'test': 1029}
